In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from collections import Counter
import numpy as np
from scipy.stats import randint

In [13]:
 
Train = pd.read_csv('Train.csv')



In [14]:
Train

,Id,OrgId,IncidentId,AlertId,DetectorId,AlertTitle,Category,IncidentGrade,EntityType,EvidenceRole,...,RegistryKey,RegistryValueName,ApplicationId,OAuthApplicationId,ResourceIdName,OSFamily,CountryCode,Day,Month,Hour
0,876173330448,19,663,2220,23,16,1,0,26,1,...,1631,635,2251,881,3586,5,242,27,5,7
1,163208759389,1526,70998,632297,85,63,5,0,22,1,...,1631,635,2251,881,3586,5,242,4,6,21
2,833223656786,44,8833,9752,50,36,7,0,16,0,...,1631,635,2251,881,3586,5,242,13,6,21
3,1030792152862,771,49554,50586,1,1,10,0,16,1,...,1631,635,2251,881,3586,5,242,9,6,2
4,1288490190139,161,31798,222463,11,9,10,0,27,0,...,1631,635,2251,881,3586,5,242,5,6,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1499198,884763263898,18,3561,56438,129,2371,7,1,9,0,...,1631,635,2251,881,3586,5,242,10,6,5
1499199,180388631256,4,287,457,8,7,10,1,14,0,...,1631,635,2251,881,3586,5,242,29,5,5
1499200,1666447311459,93,9,0,27,18,5,1,27,0,...,1631,635,2251,881,3586,5,242,12,6,1
1499201,1443109015998,105,44858,119566,6,5,10,1,15,1,...,1631,635,2251,881,3586,5,242,11,6,17


In [15]:
from sklearn.model_selection import train_test_split

# Splitting data
X= Train.drop('IncidentGrade',axis=1)
y= Train['IncidentGrade']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
# Selecting top features using anova 
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(score_func=f_classif, k=15)  # Adjust k as needed
X_new = selector.fit_transform(X_train, y_train)

selected_features = X_train.columns[selector.get_support()]
print("Selected Features:", selected_features)

Selected Features: Index(['OrgId', 'IncidentId', 'AlertId', 'DetectorId', 'AlertTitle',
       'Category', 'EntityType', 'EvidenceRole', 'Sha256', 'IpAddress',
       'AccountSid', 'DeviceName', 'NetworkMessageId', 'CountryCode', 'Day'],
      dtype='object')


In [19]:
# Keeping only the top 15 features
X_Train=X[['OrgId', 'IncidentId', 'AlertId', 'DetectorId', 'AlertTitle',
       'Category', 'EntityType', 'EvidenceRole', 'Sha256', 'IpAddress',
       'AccountSid', 'DeviceName', 'NetworkMessageId', 'CountryCode', 'Day']]
X_Train.head()

,OrgId,IncidentId,AlertId,DetectorId,AlertTitle,Category,EntityType,EvidenceRole,Sha256,IpAddress,AccountSid,DeviceName,NetworkMessageId,CountryCode,Day
0,19,663,2220,23,16,1,26,1,138268,360606,441377,153085,529644,242,27
1,1526,70998,632297,85,63,5,22,1,9479,360606,441377,153085,529644,242,4
2,44,8833,9752,50,36,7,16,0,138268,360606,441377,153085,466234,242,13
3,771,49554,50586,1,1,10,16,1,138268,360606,441377,153085,58862,242,9
4,161,31798,222463,11,9,10,27,0,138268,360606,260882,153085,529644,242,5


# ML Model

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
y= Train['IncidentGrade']
# Split they= df_cleaned_corr['IncidentGrade'] data
X_train, X_test, y_train, y_test = train_test_split(X_Train, y, test_size=0.2, random_state=42, stratify=y)

# Initialize models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# List to store metrics
metrics_list = []

# Loop through each model
for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    
    # Fit the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Store metrics in a simple dictionary
    metrics_list.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })
    
    # Print confusion matrix and classification report
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Convert metrics list to DataFrame and display
metrics_df = pd.DataFrame(metrics_list)
print("\nEvaluation Metrics Summary:")
print(metrics_df)



Evaluating Random Forest...
Confusion Matrix:
[[96407  2298  1295]
 [ 1836 97031  1105]
 [ 2323  1881 95665]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96    100000
           1       0.96      0.97      0.96     99972
           2       0.98      0.96      0.97     99869

    accuracy                           0.96    299841
   macro avg       0.96      0.96      0.96    299841
weighted avg       0.96      0.96      0.96    299841


Evaluating Logistic Regression...


C:\Users\dhars\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Confusion Matrix:
[[47774 23612 28614]
 [28953 37407 33612]
 [19728 14208 65933]]

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.48      0.49    100000
           1       0.50      0.37      0.43     99972
           2       0.51      0.66      0.58     99869

    accuracy                           0.50    299841
   macro avg       0.50      0.50      0.50    299841
weighted avg       0.50      0.50      0.50    299841


Evaluating Decision Tree...
Confusion Matrix:
[[97567  1429  1004]
 [ 1213 97747  1012]
 [  919  1212 97738]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98    100000
           1       0.97      0.98      0.98     99972
           2       0.98      0.98      0.98     99869

    accuracy                           0.98    299841
   macro avg       0.98      0.98      0.98    299841
weighted avg       0.98      0.98      0.98    29984

# Test

In [25]:
 
test = pd.read_csv('testml.csv')
test


,Id,OrgId,IncidentId,AlertId,DetectorId,AlertTitle,Category,IncidentGrade,EntityType,EvidenceRole,...,RegistryKey,RegistryValueName,ApplicationId,OAuthApplicationId,ResourceIdName,OSFamily,CountryCode,Day,Month,Hour
0,876173330448,19,663,2220,23,16,1,0,26,1,...,1631,635,2251,881,3586,5,242,27,5,7
1,163208759389,1526,70998,632297,85,63,5,0,22,1,...,1631,635,2251,881,3586,5,242,4,6,21
2,833223656786,44,8833,9752,50,36,7,0,16,0,...,1631,635,2251,881,3586,5,242,13,6,21
3,1030792152862,771,49554,50586,1,1,10,0,16,1,...,1631,635,2251,881,3586,5,242,9,6,2
4,1288490190139,161,31798,222463,11,9,10,0,27,0,...,1631,635,2251,881,3586,5,242,5,6,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1499198,884763263898,18,3561,56438,129,2371,7,1,9,0,...,1631,635,2251,881,3586,5,242,10,6,5
1499199,180388631256,4,287,457,8,7,10,1,14,0,...,1631,635,2251,881,3586,5,242,29,5,5
1499200,1666447311459,93,9,0,27,18,5,1,27,0,...,1631,635,2251,881,3586,5,242,12,6,1
1499201,1443109015998,105,44858,119566,6,5,10,1,15,1,...,1631,635,2251,881,3586,5,242,11,6,17


In [26]:
# Keeping only the top 15 features
X_test=test[['OrgId', 'IncidentId', 'AlertId', 'DetectorId', 'AlertTitle',
       'Category', 'EntityType', 'EvidenceRole', 'Sha256', 'IpAddress',
       'AccountSid', 'DeviceName', 'NetworkMessageId', 'CountryCode', 'Day']]
X_test.head()

,OrgId,IncidentId,AlertId,DetectorId,AlertTitle,Category,EntityType,EvidenceRole,Sha256,IpAddress,AccountSid,DeviceName,NetworkMessageId,CountryCode,Day
0,19,663,2220,23,16,1,26,1,138268,360606,441377,153085,529644,242,27
1,1526,70998,632297,85,63,5,22,1,9479,360606,441377,153085,529644,242,4
2,44,8833,9752,50,36,7,16,0,138268,360606,441377,153085,466234,242,13
3,771,49554,50586,1,1,10,16,1,138268,360606,441377,153085,58862,242,9
4,161,31798,222463,11,9,10,27,0,138268,360606,260882,153085,529644,242,5


# ML Model

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_test, y, test_size=0.2, random_state=42, stratify=y
)

# Initialize Decision Tree model
model = DecisionTreeClassifier(random_state=42)

print("\nEvaluating Decision Tree...")

# Train the model
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Metrics summary
metrics_df = pd.DataFrame([{
    "Model": "Decision Tree",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}])

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nEvaluation Metrics Summary:")
print(metrics_df)



Evaluating Decision Tree...
Confusion Matrix:
[[97567  1429  1004]
 [ 1213 97747  1012]
 [  919  1212 97738]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98    100000
           1       0.97      0.98      0.98     99972
           2       0.98      0.98      0.98     99869

    accuracy                           0.98    299841
   macro avg       0.98      0.98      0.98    299841
weighted avg       0.98      0.98      0.98    299841


Evaluation Metrics Summary:
           Model  Accuracy  Precision    Recall  F1 Score
0  Decision Tree  0.977358   0.977365  0.977358  0.977359
